In [0]:
%sql
-- 1. Aplicar Tags de Governança na tabela e na coluna sensível
ALTER TABLE workspace.silver_financas.clientes 
SET TAGS ('dominio' = 'financas', 'classificacao' = 'confidencial_lgpd');

ALTER TABLE workspace.silver_financas.clientes 
ALTER COLUMN cpf SET TAGS ('pii' = 'true', 'tipo_dado' = 'identificador_pessoal');

-- 2. Criar a Camada Gold (Visão Consolidada de Negócio)
CREATE SCHEMA IF NOT EXISTS workspace.gold_financas;

-- 3. Criar a tabela Gold cruzando clientes (Silver) com transações
CREATE OR REPLACE TABLE workspace.gold_financas.resumo_credito_cliente AS
SELECT 
    c.id_cliente,
    c.nome,
    c.cpf, -- já vem protegido com a máscara!
    c.renda_mensal,
    COUNT(t.id_transacao) AS total_transacoes,
    COALESCE(SUM(t.valor_transacao), 0) AS valor_total_gasto
FROM workspace.silver_financas.clientes c
LEFT JOIN workspace.legado_financas.transacoes_credito t 
    ON c.id_cliente = t.id_cliente
GROUP BY c.id_cliente, c.nome, c.cpf, c.renda_mensal;

num_affected_rows,num_inserted_rows
